
## Criação da Base de dados

Notebook dedicado para a criação de catálogo, schemas, Volumes e tabelas Delta utilizados no projeto.

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS ANP_Combustiveis;

CREATE SCHEMA IF NOT EXISTS ANP_Combustiveis.00_raw;
CREATE SCHEMA IF NOT EXISTS ANP_Combustiveis.01_bronze;
CREATE SCHEMA IF NOT EXISTS ANP_Combustiveis.02_silver;
CREATE SCHEMA IF NOT EXISTS ANP_Combustiveis.03_gold;

CREATE VOLUME IF NOT EXISTS ANP_Combustiveis.00_raw.data;


##### Camada Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS
    anp_combustiveis.01_bronze.precos_revenda (
    regiao STRING,
    uf STRING,
    municipio STRING,
    razao_social STRING,
    cnpj_revenda STRING,
    nome_rua STRING,
    numero_rua STRING,
    complemento STRING,
    bairro STRING,
    cep STRING,
    produto STRING,
    data_coleta STRING,
    valor_venda STRING,
    valor_compra STRING,
    unidade_medida STRING,
    bandeira STRING,
    arquivo_origem STRING,
    data_hora_ingestao TIMESTAMP
) USING DELTA;


CREATE TABLE IF NOT EXISTS
    anp_combustiveis.01_bronze.vendas_municipio (
    ano_referencia STRING,
    regiao STRING,
    uf STRING,
    produto STRING,
    codigo_ibge STRING,
    municipio STRING,
    volume_vendido STRING,
    arquivo_origem STRING,
    data_hora_ingestao TIMESTAMP
) USING DELTA;


CREATE TABLE IF NOT EXISTS
    anp_combustiveis.01_bronze.municipios_ibge (
    codigo_ibge BIGINT,
    municipio STRING,
    codigo_uf BIGINT,
    uf STRING,
    nome_uf STRING,
    codigo_regiao BIGINT,
    sigla_regiao STRING,
    nome_regiao STRING,
    codigo_regiao_imediata BIGINT,
    regiao_imediata STRING,
    codigo_regiao_intermediaria BIGINT,
    regiao_intermediaria STRING,
    arquivo_origem STRING,
    data_hora_ingestao TIMESTAMP
) USING DELTA;

-- Dados estaduais do IBGE preservados como texto na Bronze
CREATE TABLE IF NOT EXISTS
    anp_combustiveis.01_bronze.estados_ibge (
    codigo_uf STRING,
    uf STRING,
    nome_uf STRING,
    ano_populacao STRING,
    populacao STRING,
    ano_area STRING,
    area_km2 STRING,
    arquivo_populacao STRING,
    arquivo_area STRING,
    data_hora_ingestao TIMESTAMP
) USING DELTA;


CREATE TABLE IF NOT EXISTS
    anp_combustiveis.01_bronze.precos_revenda_fluxo (
    regiao STRING,
    uf STRING,
    municipio STRING,
    razao_social STRING,
    cnpj_revenda STRING,
    produto STRING,
    data_coleta DATE,
    tempo_evento TIMESTAMP,
    valor_venda DECIMAL(10, 3),
    unidade_medida STRING,
    arquivo_origem STRING,
    data_hora_ingestao TIMESTAMP
) USING DELTA;

##### Camada Silver

In [0]:
%sql

-- Dimensão estadual tratada
CREATE TABLE IF NOT EXISTS
    anp_combustiveis.02_silver.estados (
    codigo_uf BIGINT,
    uf STRING,
    nome_uf STRING,
    ano_populacao INT,
    populacao BIGINT,
    ano_area INT,
    area_km2 DOUBLE,
    processado_bronze_em TIMESTAMP,
    processado_silver_em TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS
    anp_combustiveis.02_silver.precos_revenda (
    regiao STRING,
    uf STRING,
    codigo_ibge BIGINT,
    municipio STRING,
    id_revenda STRING,
    razao_social STRING,
    produto STRING,
    familia_combustivel STRING,
    data_coleta DATE,
    valor_venda DECIMAL(10, 3),
    valor_compra DECIMAL(10, 3),
    unidade_medida STRING,
    bandeira STRING,
    arquivo_origem STRING,
    processado_bronze_em TIMESTAMP,
    processado_silver_em TIMESTAMP
) USING DELTA;


CREATE TABLE IF NOT EXISTS
    anp_combustiveis.02_silver.vendas_municipio (
    ano_referencia INT,
    regiao STRING,
    uf STRING,
    codigo_ibge BIGINT,
    municipio STRING,
    produto STRING,
    familia_combustivel STRING,
    volume_vendido DOUBLE,
    arquivo_origem STRING,
    processado_bronze_em TIMESTAMP,
    processado_silver_em TIMESTAMP
) USING DELTA;


CREATE TABLE IF NOT EXISTS
    anp_combustiveis.02_silver.municipios (
    codigo_ibge BIGINT,
    municipio STRING,
    codigo_uf BIGINT,
    uf STRING,
    nome_uf STRING,
    codigo_regiao BIGINT,
    sigla_regiao STRING,
    nome_regiao STRING,
    codigo_regiao_imediata BIGINT,
    regiao_imediata STRING,
    codigo_regiao_intermediaria BIGINT,
    regiao_intermediaria STRING,
    processado_bronze_em TIMESTAMP,
    processado_silver_em TIMESTAMP
) USING DELTA;

##### Tabelas Gold

In [0]:
%sql

CREATE TABLE IF NOT EXISTS
    anp_combustiveis.03_gold.precos_semanais_fluxo (
    inicio_janela TIMESTAMP,
    fim_janela TIMESTAMP,
    uf STRING,
    municipio STRING,
    produto STRING,
    preco_medio DECIMAL(10, 3),
    preco_minimo DECIMAL(10, 3),
    preco_maximo DECIMAL(10, 3),
    quantidade_observacoes BIGINT,
    processado_gold_em TIMESTAMP
) USING DELTA;

-- Indicadores de negócio
CREATE TABLE IF NOT EXISTS
    anp_combustiveis.03_gold.indicadores_estaduais (
    ano_referencia INT,
    codigo_uf BIGINT,
    uf STRING,
    nome_uf STRING,
    familia_combustivel STRING,
    preco_medio DECIMAL(10, 3),
    volume_vendido DOUBLE,
    ano_populacao INT,
    populacao BIGINT,
    ano_area INT,
    area_km2 DOUBLE,
    litros_por_habitante DOUBLE,
    litros_por_km2 DOUBLE,
    ranking_preco INT,
    ranking_consumo INT,
    ranking_consumo_habitante INT,
    ranking_consumo_area INT,
    processado_gold_em TIMESTAMP
) USING DELTA;